# Phase D — GRPO training

Trains the adaptive policy with TRL `GRPOTrainer` + LoRA. **Runtime → Change runtime type → GPU.**

### Run order (from the proposal)

1. **Smoke test** — 5 steps, tiny group. Answers only "does it execute".
2. **Go/no-go gate** — `--lambda-think 0.0`, i.e. correctness + format, no length penalty.
   If GRPO cannot improve plain correctness over base, the adaptive question is moot and the
   project pivots to characterising why.
3. **λ sweep** — only after the gate passes.

### What to watch

`metric_think_rate` in the logs, not the reward curve. Collapse to all-think or all-no-think
is the primary failure mode and it is invisible in reward alone — a policy that stopped
reasoning and one that started emitting malformed calls both flatten it.

From the fp16 baselines: the **oracle thinks on 17.4%** of items, prompting alone gives 96.9%.
A trained policy near 17–20% is in the right regime; near 0% or near 100% is collapse.

### Why λ is swept where it is

Break-even λ per category, computed from the fp16 baselines: `simple_python` never worth
thinking (Δ = −2.8%), `multiple` 0.07, `parallel` 0.48, `parallel_multiple` 0.53,
`irrelevance` 2.11. Four of five switch off below 0.55, then nothing changes until 2.11 —
so `{0.05, 0.1, 0.25, 0.5, 1.0, 2.0}` covers every transition and a linear sweep would not.

## 1 — Install

⚠️ **A session restart after this cell is mandatory, not optional.** Colab does not always offer
the button. Run **Runtime → Restart session** yourself, then continue at step 2. Skipping it makes
the verify cell report stale versions and look as though the installs silently failed.

Six steps, and **the order matters** — every one is a lesson from `notes/engineering_log.md`:

1. **vLLM first** (entry 9). Rollout generation, not the optimizer, is what makes a GRPO step
   expensive. With HF `generate` a step at 16 completions x 768 tokens runs several minutes, so
   200 steps does not fit a Colab session. vLLM's continuous batching is the same ~20x that turned
   a 16 h baseline run into 36 min. It also has the strictest torch pin, so it goes first.
2. **torch from the cu130 index** (entry 10). vLLM 0.26 ships a CUDA 13 binary; Colab preinstalls
   `torch 2.11.0+cu128`. pip treats the version `2.11.0` as satisfying vLLM's pin and leaves torch
   alone, so a cu13 `.so` hunts for `libcudart.so.13` on a cu12 system.
3. **Restore numpy** (entry 16). `--index-url` *replaces* PyPI rather than adding to it, so step 2
   drags torch's whole dependency tree off the PyTorch index — including numpy 2.0.2, which breaks
   every compiled extension built against 2.1+.
4. **trl / peft / datasets**, after the torch repair so they resolve against the torch that will
   actually be used.
5. **Uninstall torchao** (entry 13). Nothing uses it, but PEFT's LoRA dispatcher calls
   `is_torchao_available()`, which *raises* on Colab's 0.10.0 instead of returning False.
6. **bfcl-eval `--no-deps`** (entries 7 and 14). Training runs the checker once per rollout, but
   `scoring.bfcl_scorer` supplies the one boolean its 81-package import chain exists to resolve.

The cell imports nothing heavy on purpose (entry 17) and ends with a subprocess check, which reads
what is genuinely on disk rather than what this kernel has cached.

In [ ]:
# NOTHING heavy is imported in this cell, deliberately. `import torch` here would
# pull torch AND numpy into the kernel, and every pip command afterwards would
# then change files on disk that the kernel has already cached — making the
# verify cell report stale versions and look like the installs silently failed.
# importlib.metadata reads package metadata WITHOUT importing the package.
from importlib.metadata import version

V = version("torch").split("+")[0]
print("torch on disk before repair:", version("torch"))

# 1. vLLM first — it has the strictest torch pin.
!pip install -q vllm

# 2. Repair the CUDA ABI: same torch VERSION, cu130 build (engineering log entry 10).
!pip install -q --force-reinstall torch=={V} torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

# 3. Undo the collateral damage from step 2 (entry 16). `--index-url` REPLACES PyPI
#    rather than adding to it, so --force-reinstall pulled torch's whole dependency
#    tree off the PyTorch index — including numpy 2.0.2, which breaks every compiled
#    extension built against 2.1+ (`_blas_supports_fpe` arrived in 2.1).
!pip install -q -U "numpy>=2.2" --index-url https://pypi.org/simple

# 4. NVRTC, for vLLM's cumem allocator (entries 19 and 20). Without it sleep mode
#    fails with "cumem allocator is not supported on current platform", and without
#    sleep mode vLLM's ~5 GiB stays pinned through the backward pass and a T4 OOMs.
!pip install -q nvidia-cuda-nvrtc

# 5. Training stack, after the torch repair so these resolve their compiled deps
#    against the torch that will actually be used. They accept any torch 2.x.
!pip install -q trl peft datasets accelerate

# 6. PEFT's LoRA dispatcher raises on Colab's old torchao instead of skipping it (entry 13).
!pip uninstall -y -q torchao

# 7. BFCL data + checker, without its 81-package handler chain (entries 7 and 14).
!pip install -q --no-deps bfcl-eval==2026.3.23

print("\n--- on disk now ---")
!python -c "import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__)"
print("\n>>> RESTART SESSION now (Runtime -> Restart session), then run step 2. <<<")

## 2 — Verify, then get the code

In [ ]:
import glob
import importlib.util
import os

import numpy
import torch

# numpy is imported and printed first on purpose: a downgraded numpy breaks the
# compiled extensions inside transformers and vLLM, and the resulting traceback
# blames "Could not import module 'AutoModel'" rather than numpy (entry 16).
print("numpy     :", numpy.__version__)
print("torch     :", torch.__version__, "| CUDA", torch.version.cuda)

# NVIDIA wheels drop .so files under site-packages/nvidia/<component>/lib, and may
# ship libnvrtc.so.13.x.y without the bare soname the linker looks for. vLLM's
# cumem allocator — and therefore sleep mode, and therefore fitting on a T4 at all
# — needs it resolvable (entries 19, 20).
hits = sorted(glob.glob("/usr/local/lib/python3.12/dist-packages/nvidia/**/libnvrtc.so*", recursive=True))
for path in hits:
    soname = os.path.join(os.path.dirname(path), "libnvrtc.so.13")
    if not os.path.exists(soname) and ".so.13." in path:
        os.symlink(path, soname)
        print("symlinked :", soname, "->", os.path.basename(path))
if hits:
    libdirs = sorted({os.path.dirname(p) for p in hits})
    os.environ["LD_LIBRARY_PATH"] = ":".join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")])

import bfcl_eval
import peft
import transformers
import trl
import vllm
from transformers import AutoModelForCausalLM  # fails first on a numpy mismatch

data_dir = os.path.join(os.path.dirname(bfcl_eval.__file__), "data")
print("vllm      :", vllm.__version__)
print("trl       :", trl.__version__)
print("peft      :", peft.__version__)
print("transformers:", transformers.__version__)
print("bfcl data :", os.path.isdir(data_dir))
print("bf16      :", torch.cuda.is_bf16_supported(), "(False on T4 — fp16 fallback)")
print("torchao   :", "absent (good)" if importlib.util.find_spec("torchao") is None else "PRESENT — rerun the uninstall")

assert os.path.isdir(data_dir), "BFCL data missing — rerun the --no-deps install"
assert importlib.util.find_spec("torchao") is None, "torchao still installed; PEFT will raise on LoRA injection"

# The real test of the NVRTC fix. Without this, --vllm-sleep dies at construction.
!python -c "import vllm.cumem_allocator; print('cumem    : OK (sleep mode available)')"

print("\nenvironment OK")
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
REPO = "https://github.com/widodu77/rs_aidams.git"

if os.path.isdir("/content/rs_aidams"):
    !cd /content/rs_aidams && git pull --ff-only
else:
    !git clone -q $REPO /content/rs_aidams

%cd /content/rs_aidams
os.environ["PYTHONPATH"] = "src"
!git log --oneline -1

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_RUNS = "/content/drive/MyDrive/rs_aidams/runs"
os.makedirs(DRIVE_RUNS, exist_ok=True)

# Informational only. The memory-shaped settings (forward batch, vLLM reservation,
# sleep mode) are resolved inside train_grpo.py from the GPU it actually finds —
# they used to be computed here and passed as a $FLAGS string, which failed
# silently when the notebook was updated but the kernel kept the old variable.
name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.mem_get_info()[1] / 2**30
cc = torch.cuda.get_device_capability(0)
print(f"{name}  |  {total_gib:.1f} GiB  |  sm_{cc[0]}{cc[1]}")
print("bf16:", torch.cuda.is_bf16_supported())

STEPS = 100 if cc[0] >= 8 else 50
print(f"\nsteps per lambda: {STEPS}   (~85 s/step on a T4; faster on Ada)")
print("training config is auto-detected per GPU — see the line train_grpo.py prints at startup")

## 3 — Smoke test

Five steps at the real completion length (768), rollouts through vLLM.

Memory on a T4 is genuinely tight, so the shape of this run is chosen rather than default:
vLLM colocate holds its own copy of the weights plus KV cache, the training model holds another,
and the log-probability forward materialises a `batch x tokens x 151936` logits tensor **twice**
(policy and reference). That last term is what OOM'd the first attempt — it is why the forward
batch is 2 rather than 8, and why vLLM sleeps between generation phases.

Three things to read off:

- **`step_time`** — the feasibility gate. Multiply by 200 for one run, then by 7 for gate plus six
  λ values. Near 30 s/step means ~1.7 h per run; much above 60 s/step and we cut steps or group
  size before committing GPU hours.
- **`frac_reward_zero_std`** — fraction of groups where every rollout scored identically. Those
  steps produce zero advantage and therefore zero gradient. It was 1.0 on three of five steps at
  group size 4; group size 8 should reduce it.
- **peak memory** — if it OOMs again, the levers in order are
  `--vllm-gpu-memory-utilization 0.25`, then `--per-device-batch-size 1`.

Read nothing scientific into five steps.

In [ ]:
# No hardware flags here — train_grpo.py detects the GPU and prints what it chose.
# Five steps is enough to catch an OOM in ~2 minutes instead of 40 minutes into a sweep.
!python -m train.train_grpo --output runs/smoke --lambda-think 0.0 --use-vllm \
    --max-steps 5 --num-generations 8 --max-completion-length 768

## 4 — Go/no-go gate (λ = 0)

**50 steps (~1.2 h), and read it as a mechanism check rather than a verdict.**

The proposal specifies this gate as "train on correctness reward only; does accuracy improve over
base?" The first full smoke run showed why that is the *hardest* configuration to learn from, not
the neutral one:

- `frac_reward_zero_std` was 1.0 on three of five steps, **even at group size 8** — all eight
  rollouts correct, reward 1.2 flat, `grad_norm 0`.
- With only correctness and format the reward takes very few distinct values, and a model already
  at 87% accuracy saturates groups constantly.
- But `metric_mean_think_tokens/std` on those same steps was 57.4, 45.1, 11.1 — the rollouts
  differ in reasoning *length* even when they agree on correctness. **Any λ > 0 turns those dead
  groups into informative ones.**

So weak movement here is expected and is **not** evidence that GRPO cannot work on this task. What
this run has to establish is narrower: that the loop is sound, the reward behaves, and the
zero-variance rate is what the smoke test predicted. The λ sweep is where learning is expected.

Baseline to beat, on the **eval split only** — the split manifest is written into the run directory
so the fixed-policy baselines can be restricted to the same items: adaptive-prompt at 87.5%
accuracy and 287.5 mean tokens.

In [ ]:
# Hardware settings are auto-detected inside the script. --resume is safe on a fresh
# run and picks up the latest checkpoint otherwise, so this cell can simply be re-run
# after any interruption.
!python -m train.train_grpo --output runs/gate --lambda-think 0.0 --use-vllm \
    --max-steps 50 --resume
!cp -r runs/gate $DRIVE_RUNS/

In [ ]:
# Training traces. Watch think-rate and zero-variance, not the reward curve.
import json


def show(run="runs/gate"):
    history = json.load(open(f"{run}/log_history.json", encoding="utf-8"))
    rows = [r for r in history if "reward" in r]
    if not rows:
        print("no training rows logged")
        return

    cols = [
        ("step", "step", "{:>6}"),
        ("reward", "reward", "{:>8.3f}"),
        ("think", "rewards/metric_think_rate/mean", "{:>8.3f}"),
        ("correct", "rewards/metric_correctness/mean", "{:>8.3f}"),
        ("format", "rewards/metric_format_rate/mean", "{:>8.3f}"),
        ("thinkTok", "rewards/metric_mean_think_tokens/mean", "{:>9.1f}"),
        # Fraction of groups where every rollout scored the same -> zero
        # advantage -> zero gradient. A step like that teaches nothing.
        ("zeroStd", "frac_reward_zero_std", "{:>8.2f}"),
        # Fraction of rollouts that hit the completion cap. These never reached
        # a tool call, so they score zero correctness by construction.
        ("clipped", "completions/clipped_ratio", "{:>8.2f}"),
        ("entropy", "entropy", "{:>8.3f}"),
    ]

    print("".join(f"{name:>9s}" for name, _, _ in cols))
    for r in rows[:: max(1, len(rows) // 25)]:
        out = []
        for _, key, fmt in cols:
            v = r.get(key)
            out.append(fmt.format(v) if isinstance(v, (int, float)) else f"{'-':>8s}")
        print("".join(out))

    dead = [r for r in rows if r.get("frac_reward_zero_std", 0) == 1.0]
    print(f"\nsteps with no gradient at all: {len(dead)}/{len(rows)}")
    print("oracle think-rate is 0.174; near 0 or near 1 is collapse")


show()

## 5 — λ sweep (only after the gate passes)

**Five values × 100 steps ≈ 12 h**, at the measured ~85 s/step. Each run is independent and skips
itself if its adapter is already on Drive, so a session drop costs one λ rather than the sweep, and
you can spread this across sessions.

`{0.05, 0.25, 0.5, 1.0, 2.0}` covers every break-even transition computed from the fp16 baselines
— `multiple` at 0.07, `parallel` 0.48, `parallel_multiple` 0.53, `irrelevance` 2.11, with
`simple_python` never worth thinking (Δ = −2.8%). Five rough points beat three polished ones for a
Pareto curve; a linear sweep to 3.0 would put half its points in dead space.

The prediction to test: as λ rises, thinking should switch off in that order — `simple_python`
first, then `multiple`, then the parallel categories, with `irrelevance` last. That is falsifiable
independently of where the curve lands, and it connects straight to H1.

In [ ]:
for lam in [0.05, 0.25, 0.5, 1.0, 2.0]:
    tag = f"lam{lam}".replace(".", "_")
    if os.path.exists(f"{DRIVE_RUNS}/{tag}/adapter_model.safetensors"):
        print(f"skipping {tag}, already done")
        continue
    print(f"\n{'='*70}\nlambda = {lam}  ({STEPS} steps)\n{'='*70}", flush=True)
    # Restore any partial run from Drive first, so --resume can continue it after a
    # session drop rather than restarting the lambda from zero.
    !mkdir -p runs/$tag && cp -rn $DRIVE_RUNS/$tag/. runs/$tag/ 2>/dev/null
    !python -m train.train_grpo --output runs/$tag --lambda-think $lam --use-vllm \
        --max-steps $STEPS --resume
    !cp -r runs/$tag $DRIVE_RUNS/
    # The checkpoint that matters, and it is readable at the FIRST lambda: the gate at
    # lambda=0 had 37/50 steps with no gradient. If lambda>0 does not collapse that
    # toward zero, the reward model is wrong and the sweep should stop here.
    import json as _json
    _h = [r for r in _json.load(open(f"runs/{tag}/log_history.json", encoding="utf-8")) if "reward" in r]
    _dead = sum(1 for r in _h if r.get("frac_reward_zero_std", 0) == 1.0)
    print(f"lambda={lam}: {_dead}/{len(_h)} steps with no gradient "
          f"({_dead/max(len(_h),1):.0%}; lambda=0 was 74%)", flush=True)

## 6 — Evaluate the trained policies

Generates each adapter's output on the **held-out eval split only** (248 items), through
`generate.run_vllm` — the same path that produced the fixed-policy baselines. Scoring a trained
policy through a different pipeline than its baselines is how comparisons quietly break.

Two things are pinned rather than defaulted, and both are load-bearing:

- **`--dtype half`.** The baselines were generated in fp16 because a T4 has no bf16. On an L4
  `--dtype auto` resolves to bf16, and a policy evaluated in bf16 against fp16 baselines is not a
  clean comparison. This project has already had two findings overturned by a precision change
  (NF4 vs fp16, `notes/2026-08-09.md`), so precision is fixed deliberately.
- **`--split-manifest ... --split eval`.** Every run wrote the same manifest (same seed, same
  stratification), so all adapters and all baselines are compared on identical held-out items.
  Without it a trained policy would be scored partly on what it trained on.

Decoding is greedy, matching the baselines, so any difference reflects the policy rather than
sampling noise.

In [ ]:
import os

EVAL_DIR = "results/raw/vllm_eval"
DRIVE_EVAL = "/content/drive/MyDrive/rs_aidams/results/raw/vllm_eval"
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(DRIVE_EVAL, exist_ok=True)

# Every run wrote an identical manifest (same seed, same stratification), so any of
# them defines the eval split. Using one explicitly makes the dependency visible.
MANIFEST = "runs/lam0_05/split_manifest.json"
assert os.path.exists(MANIFEST), "run the sweep first — the manifest is written by train_grpo"

for lam in [0.05, 0.25, 0.5, 1.0, 2.0]:
    tag = f"lam{lam}".replace(".", "_")
    out = f"{EVAL_DIR}/trained_{tag}.jsonl"
    if os.path.exists(f"{DRIVE_EVAL}/trained_{tag}.jsonl"):
        print(f"skipping {tag}, already evaluated")
        continue
    if not os.path.exists(f"runs/{tag}/adapter_model.safetensors"):
        !cp -r $DRIVE_RUNS/$tag runs/ 2>/dev/null
    print(f"\n{'='*70}\nevaluating lambda = {lam}\n{'='*70}", flush=True)
    !python -m generate.run_vllm --policy adaptive --adapter runs/$tag --dtype half \
        --split-manifest $MANIFEST --split eval --out $out
    !cp $out $DRIVE_EVAL/

print("\ndone — copy results/raw/vllm_eval/ down and score locally")

## Then, locally

Copy `results/raw/vllm_eval/` and the split manifest down from Drive, then score everything on
the **same held-out items** — trained policies and fixed-policy baselines alike:

```bash
uv run python -m analysis.score_run \
    results/raw/vllm_eval/*.jsonl \
    results/raw/vllm/qwen3-1.7b_never.jsonl \
    results/raw/vllm/qwen3-1.7b_always.jsonl \
    results/raw/vllm/qwen3-1.7b_adaptive.jsonl \
    --split-manifest runs/lam0_05/split_manifest.json --split eval \
    --out results/eval_metrics.json
```

The baselines were generated over all 1240 items, so `--split-manifest` is what restricts them to
the 248 the policies never trained on. That single flag is the difference between a comparison and
a leak.

Reference points on that eval set: `never`, `always`, the `adaptive` prompt (87.5% at 287 tokens
on the full set), and the per-item oracle (91.2% at 97 tokens) as the ceiling.